## Step 1: Stimulus indexer + validator

In [1]:
import json
from pathlib import Path
from collections import defaultdict, Counter

# =========================
# CONFIG
# =========================
STIMULI_DIR = Path("stimuli_out")          # <- change if needed
OUTPUT_DIR  = Path("design_out")           # index file destination
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_PRODUCTS = [f"P{i}" for i in range(1, 7)]
EXPECTED_VARIANTS = ["GENERIC", "E_PLUS", "E_MINUS", "O_PLUS", "O_MINUS"]

# =========================
# HELPERS
# =========================
def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def extract_stimulus_fields(obj: dict, path: Path) -> dict:
    """
    Canonical field paths from your confirmed stimulus structure:
      stimulus_id      = id.output_id
      condition_id     = id.condition_id
      product_id       = meta.product_id
      product_name     = meta.product_name
      trait_axis       = meta.trait_axis
      variant          = meta.variant
      paraphrase_id    = meta.paraphrase_id
    """
    try:
        stimulus_id = obj["id"]["output_id"]
        condition_id = obj["id"]["condition_id"]
        meta = obj["meta"]
        product_id = meta["product_id"]
        product_name = meta.get("product_name")
        trait_axis = meta.get("trait_axis")
        variant = meta["variant"]
        paraphrase_id = meta.get("paraphrase_id")
    except KeyError as e:
        raise KeyError(f"Missing key {e} in {path}")

    rec = {
        "stimulus_id": stimulus_id,
        "condition_id": condition_id,
        "filepath": str(path.as_posix()),
        "product_id": product_id,
        "product_name": product_name,
        "trait_axis": trait_axis,
        "variant": variant,
        "paraphrase_id": paraphrase_id,
        # include generation fields for provenance if present
        "generation_model": obj.get("generation", {}).get("model"),
        "generation_temperature": obj.get("generation", {}).get("temperature"),
        "generation_timestamp_utc": obj.get("generation", {}).get("timestamp_utc"),
    }
    return rec

# =========================
# SCAN + INDEX
# =========================
if not STIMULI_DIR.exists():
    raise FileNotFoundError(f"STIMULI_DIR not found: {STIMULI_DIR.resolve()}")

files = sorted(STIMULI_DIR.glob("*.json"))
print(f"Found {len(files)} JSON files in {STIMULI_DIR.resolve()}")

records = []
errors = []

for fp in files:
    try:
        obj = load_json(fp)
        rec = extract_stimulus_fields(obj, fp)
        records.append(rec)
    except Exception as e:
        errors.append((str(fp), str(e)))

if errors:
    print("\n=== ERRORS (files that failed to parse) ===")
    for fp, msg in errors[:20]:
        print(f"- {fp}: {msg}")
    if len(errors) > 20:
        print(f"... and {len(errors)-20} more")
    print("\nFix these before proceeding, otherwise you may get missing variants later.\n")

# Build indices
index_pv = defaultdict(list)    # (product_id, variant) -> [records...]
index_pvp = defaultdict(list)   # (product_id, paraphrase_id, variant) -> [records...]

for r in records:
    index_pv[(r["product_id"], r["variant"])].append(r)
    index_pvp[(r["product_id"], r["paraphrase_id"], r["variant"])].append(r)

# =========================
# VALIDATION REPORT
# =========================
print("\n=== VALIDATION: coverage per product/variant ===")
missing = []
duplicates = []

for pid in EXPECTED_PRODUCTS:
    for var in EXPECTED_VARIANTS:
        key = (pid, var)
        n = len(index_pv.get(key, []))
        if n == 0:
            missing.append((pid, var))
        elif n > 1:
            duplicates.append((pid, var, n))

# Summary table-ish output
for pid in EXPECTED_PRODUCTS:
    counts = {var: len(index_pv.get((pid, var), [])) for var in EXPECTED_VARIANTS}
    counts_str = "  ".join([f"{var}:{counts[var]}" for var in EXPECTED_VARIANTS])
    print(f"{pid}: {counts_str}")

if missing:
    print("\n=== MISSING (product_id, variant) ===")
    for pid, var in missing:
        print(f"- {pid} missing {var}")

if duplicates:
    print("\n=== DUPLICATES (product_id, variant, count) ===")
    for pid, var, n in duplicates:
        print(f"- {pid} has {n} files for {var}")
        # print filepaths to help you delete/resolve
        for rec in index_pv[(pid, var)][:5]:
            print(f"    {rec['filepath']}")
        if n > 5:
            print(f"    ... (+{n-5} more)")

# Optional: sanity checks about trait_axis consistency
# E variants should typically have meta.trait_axis == "EXTRAVERSION"
# O variants should typically have meta.trait_axis == "OPENNESS"
inconsistent = []
for r in records:
    v = r["variant"]
    ta = r["trait_axis"]
    if v.startswith("E_") and ta not in (None, "EXTRAVERSION"):
        inconsistent.append((r["product_id"], v, ta, r["filepath"]))
    if v.startswith("O_") and ta not in (None, "OPENNESS"):
        inconsistent.append((r["product_id"], v, ta, r["filepath"]))

if inconsistent:
    print("\n=== TRAIT_AXIS INCONSISTENCIES (not fatal, but check) ===")
    for pid, v, ta, fp in inconsistent[:20]:
        print(f"- {pid} {v}: trait_axis={ta}  file={fp}")
    if len(inconsistent) > 20:
        print(f"... and {len(inconsistent)-20} more")

# =========================
# SAVE INDEX (JSON)
# =========================
# Save as list-of-records and also as a resolved lookup (product_id->variant->single record)
# If duplicates exist, we DO NOT pick one automatically: we store all and you resolve duplicates first.

index_out = {
    "meta": {
        "stimuli_dir": str(STIMULI_DIR.resolve().as_posix()),
        "n_files": len(files),
        "n_parsed": len(records),
        "n_errors": len(errors),
        "expected_products": EXPECTED_PRODUCTS,
        "expected_variants": EXPECTED_VARIANTS,
    },
    "records": records,
    "index_pv": {
        f"{pid}::{var}": recs for (pid, var), recs in index_pv.items()
    },
    "index_pvp": {
        f"{pid}::{par}::{var}": recs for (pid, par, var), recs in index_pvp.items()
    },
}

out_path = OUTPUT_DIR / "stimuli_index.json"
with out_path.open("w", encoding="utf-8") as f:
    json.dump(index_out, f, ensure_ascii=False, indent=2)

print(f"\nSaved index to: {out_path.resolve()}")
print("\nNext: if missing/duplicates are clean, we generate persona templates + participant registry + design schedules.")

Found 78 JSON files in C:\Users\alec.traub\OneDrive\Masters\031_Modeling_Communication_and_Abstraction_Part_I\Final project\Code\stimuli_out

=== VALIDATION: coverage per product/variant ===
P1: GENERIC:1  E_PLUS:3  E_MINUS:3  O_PLUS:3  O_MINUS:3
P2: GENERIC:1  E_PLUS:3  E_MINUS:3  O_PLUS:3  O_MINUS:3
P3: GENERIC:1  E_PLUS:3  E_MINUS:3  O_PLUS:3  O_MINUS:3
P4: GENERIC:1  E_PLUS:3  E_MINUS:3  O_PLUS:3  O_MINUS:3
P5: GENERIC:1  E_PLUS:3  E_MINUS:3  O_PLUS:3  O_MINUS:3
P6: GENERIC:1  E_PLUS:3  E_MINUS:3  O_PLUS:3  O_MINUS:3

=== DUPLICATES (product_id, variant, count) ===
- P1 has 3 files for E_PLUS
    stimuli_out/P1_E_PLUS_p1_gpt-4o_3371e94acf164ac9.json
    stimuli_out/P1_E_PLUS_p2_gpt-4o_437c829d7613e497.json
    stimuli_out/P1_E_PLUS_p3_gpt-4o_bb8aec86ecea4a98.json
- P1 has 3 files for E_MINUS
    stimuli_out/P1_E_MINUS_p1_gpt-4o_8834de373fd89fbf.json
    stimuli_out/P1_E_MINUS_p2_gpt-4o_2b955b6cdee40b3b.json
    stimuli_out/P1_E_MINUS_p3_gpt-4o_7c41213e67bb4b66.json
- P1 has 3 files